# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/menna890/-Explaining-Search-Performance-Gaps-Using-Ranking-Signals/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My Lane as an ML Task

**Task type: Ranking**

I chose ranking because the final output is a prioritized list of pages.
A content writer needs to know "which 20 pages first" — not just
"these 300 are broken."

**Why not classification?**
Binary yes/no hides the difference between a page missing 5 clicks
and one missing 500 clicks. The writer needs priority order.

**Why not clustering?**
I already know what I am looking for: underperforming pages.
I am not discovering unknown groups.

In [3]:

import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")


Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


In [4]:
import pandas as pd
import numpy as np

# Load starter data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Validate: ranking makes sense because position_tier exists and has clear tiers
print("Position tier distribution:")
print(df["position_tier"].value_counts())

# Validate: top_3 pages have wide range of clicks → ranking needed, not just binary
top3 = df[df["position_tier"] == "top_3"].copy()
top3 = top3[top3["impressions_90d"] >= 100].copy()

print(f"\nTop-3 pages with traffic: {len(top3):,}")
print(f"Clicks range: {top3['clicks_90d'].min()} to {top3['clicks_90d'].max()}")
print(f"CTR range: {top3['ctr'].min():.2f}% to {top3['ctr'].max():.2f}%")
print("\n→ Wide spread in performance = ranking (not just yes/no) is the right task.")

Position tier distribution:
position_tier
page_1      11814
striking     7304
page_3_5     7242
top_3        2321
deep         1319
Name: count, dtype: int64

Top-3 pages with traffic: 533
Clicks range: 0 to 2689
CTR range: 0.00% to 6.19%

→ Wide spread in performance = ranking (not just yes/no) is the right task.


## 2. Target or Proxy

**What I predict: `underperformance_score`**

The score = expected clicks − actual clicks.

- Expected clicks come from: impressions × typical CTR for the position tier.
- Actual clicks come from: measured `clicks_90d` in the data.
- The gap is computed from observed numbers, not from someone's rule.

**Binary proxy for training:**
`is_underperforming` = 1 if actual < 50% of expected, else 0.
This helps train the model, but the final output is the continuous score.

In [5]:


top3 = df[df["position_tier"] == "top_3"].copy()
top3 = top3[top3["impressions_90d"] >= 100].copy()

# Expected clicks = impressions × typical CTR (observed from data)
typical_ctr = top3["clicks_90d"].sum() / top3["impressions_90d"].sum()
top3["expected_clicks"] = top3["impressions_90d"] * typical_ctr

# Continuous target: gap size
top3["underperformance_score"] = (top3["expected_clicks"] - top3["clicks_90d"]).clip(lower=0)

# Binary proxy for training
top3["is_underperforming"] = (top3["clicks_90d"] / top3["expected_clicks"]) < 0.5

print(f"Typical CTR: {typical_ctr*100:.2f}%")
print(f"Underperforming: {top3['is_underperforming'].sum()} pages")
print("\nTarget columns (first 5 rows):")
print(top3[["content_id", "clicks_90d", "expected_clicks",
            "underperformance_score", "is_underperforming"]].head())

Typical CTR: 0.49%
Underperforming: 303 pages

Target columns (first 5 rows):
               content_id  clicks_90d  expected_clicks  \
10   content_d8ee6cc6d642         324       101.903441   
43   content_1938955b34c4           0         0.896326   
88   content_998f6f88784c           3        58.714192   
161  content_02bcf3eec147           0         1.432172   
242  content_af41d1db999a           3         7.292387   

     underperformance_score  is_underperforming  
10                 0.000000               False  
43                 0.896326                True  
88                55.714192                True  
161                1.432172                True  
242                4.292387                True  


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [7]:
K = 50

# 1. Random ranking
random_sample = top3.sample(n=K, random_state=42)
random_precision = random_sample["is_underperforming"].mean()

# 2. Fixed rule: flag pages with CTR below median
threshold = top3["ctr"].quantile(0.75)
rule_candidates = top3[top3["ctr"] < threshold].nlargest(K, "impressions_90d")
rule_precision = rule_candidates["is_underperforming"].mean()

# 3. Perfect ranking (upper bound)
perfect_candidates = top3.nlargest(K, "underperformance_score")
perfect_precision = perfect_candidates["is_underperforming"].mean()

print(f"Precision@{K}:")
print(f"  Random:  {random_precision:.3f}")
print(f"  Rule:    {rule_precision:.3f}")
print(f"  Perfect: {perfect_precision:.3f}")
print(f"\nMy model should beat {rule_precision:.3f}, aim for >0.50")

Precision@50:
  Random:  0.600
  Rule:    0.560
  Perfect: 0.820

My model should beat 0.560, aim for >0.50


## 4. The Unit of Analysis

**One row = one content item (one page)**

The dataframe below shows the grain: each row represents a single pseudonymized content item, with its search performance metrics and content-quality signals.

**Why this grain?**
- The decision is "which page to fix," so the unit of prediction must be the page.
- A coarser grain (e.g., one row per client) would lose page-level signal.
- A finer grain (e.g., one row per day) would create leakage and duplication.

**Key columns in the slice:**
- Identifiers: `content_id`, `client_id` (for grouping only, never as features)
- Performance: `impressions_90d`, `clicks_90d`, `ctr`, `avg_position`, `position_tier`
- Signals: `word_count`, `content_age_days`, `days_since_last_update`, `engagement_rate`, `scroll_rate`
- Target (computed): `underperformance_score`, `is_underperforming`

In [8]:
# Show the unit of analysis: one row = one content item (page)

# Lane slice: top-3 pages with traffic + target columns
lane_df = top3[["content_id", "client_id", "position_tier",
                "impressions_90d", "clicks_90d", "ctr",
                "word_count", "content_age_days", "days_since_last_update",
                "engagement_rate", "scroll_rate", "main_intent", "content_type",
                "underperformance_score", "is_underperforming"]].copy()

print(f"Dataframe shape: {lane_df.shape}")
print(f"→ {lane_df.shape[0]:,} rows = {lane_df.shape[0]:,} content items (pages)")
print(f"→ {lane_df.shape[1]} columns = signals + target\n")

print("First 3 rows (the unit of analysis):")
print(lane_df.head(3).T)  # Transpose for readability

print(f"\nGrain check: unique content_ids = {lane_df['content_id'].nunique():,}")
print(f"Matches row count? {lane_df['content_id'].nunique() == len(lane_df)}")

Dataframe shape: (533, 15)
→ 533 rows = 533 content items (pages)
→ 15 columns = signals + target

First 3 rows (the unit of analysis):
                                          10                    43  \
content_id              content_d8ee6cc6d642  content_1938955b34c4   
client_id                  client_19581e27de     client_f369cb89fc   
position_tier                          top_3                 top_3   
impressions_90d                        20919                   184   
clicks_90d                               324                     0   
ctr                                     1.55                   0.0   
word_count                               NaN                2394.0   
content_age_days                         329                   138   
days_since_last_update                   104                    20   
engagement_rate                         6.75                   0.0   
scroll_rate                             9.55                   0.0   
main_intent             

## 5. Why ML Beats a Fixed Rule

I tested a fixed rule (CTR below top-3's 75th percentile) across three position tiers.
Same threshold, very different results:

In [21]:

fixed_threshold = top3["ctr"].quantile(0.75)

for tier in ["top_3", "page_1", "page_3_5"]:
    tier_df = df[(df["position_tier"] == tier) & (df["impressions_90d"] >= 100)].copy()
    if len(tier_df) == 0:
        continue

    # True underperforming
    t_ctr = tier_df["clicks_90d"].sum() / tier_df["impressions_90d"].sum()
    tier_df["expected"] = tier_df["impressions_90d"] * t_ctr
    tier_df["true_bad"] = (tier_df["clicks_90d"] / tier_df["expected"]) < fixed_threshold

    # Fixed rule
    tier_df["rule_bad"] = tier_df["ctr"] < fixed_threshold

    hits = (tier_df["rule_bad"] & tier_df["true_bad"]).sum()
    total_flagged = tier_df["rule_bad"].sum()
    precision = hits / total_flagged if total_flagged > 0 else 0

    print(f"{tier:12s}: precision = {precision:.3f} (flagged {total_flagged:,})")


top_3       : precision = 0.750 (flagged 396)
page_1      : precision = 0.515 (flagged 6,601)
page_3_5    : precision = 0.570 (flagged 5,627)




| Tier | Precision | Pages Flagged |
|---|---|---|
| top_3 | 0.750 | 396 |
| page_1 | **0.515** | **6,601** |
| page_3_5 | 0.570 | 5,627 |

**Key finding:** The rule works for top_3 (75% precision) but fails for page_1
(51.5% — worse than random guessing). A content writer following this rule
for page_1 pages would do better by picking pages at random.

**Why?** A fixed threshold ignores context:
- What's "low CTR" for top_3 is normal for page_3_5
- page_1 has different baseline CTR than top_3
- Content type, intent, and client also vary

**ML wins because** it learns per-tier, per-context boundaries automatically.
It does not force one threshold on every page.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.